# Kraken Spot-Perpetual Arbitrage Screening

Mirrors the approach in `binance_arbitrage_test.ipynb` but targets Kraken spot vs Kraken Futures perpetual (PF_) markets.

**Approach:**
1. Discover all Kraken spot pairs and Kraken Futures perpetuals, find overlaps ranked by volume.
2. Download 1-minute OHLC for one calibration week and one trading week, cached as Parquet.
3. Compute spread (perp − spot) / spot × 10,000 bps per candle.
4. Calibrate an entry threshold per pair using the first week.
5. Run an out-of-sample simulation on the second week and rank pairs by net PnL.

**Fee model (taker / taker):**
- Kraken spot taker: 0.26 %
- Kraken Futures taker: 0.05 %
- Taker round-trip: ≈ 62 bps — higher than Binance (30 bps), so minimum viable spread is wider.

In [11]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import requests
from IPython.display import display
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

In [ ]:
# ── Cache directory ────────────────────────────────────────────────────────────
DATA_DIR = Path("notebooks/kraken/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# ── API base URLs ──────────────────────────────────────────────────────────────
KRAKEN_SPOT_BASE = "https://api.kraken.com/0/public"
KRAKEN_FUTURES_BASE = "https://futures.kraken.com/derivatives/api/v3"

# ── Date range ─────────────────────────────────────────────────────────────────
CALIBRATION_WEEK_START_ISO = "2026-05-26T00:00:00+00:00"
CALIBRATION_WEEK_END_ISO   = "2026-06-01T23:59:00+00:00"
CALIBRATION_WEEK_END_EXCLUSIVE_ISO = "2026-06-02T00:00:00+00:00"
TRADING_WEEK_START_ISO     = "2026-06-02T00:00:00+00:00"
TRADING_WEEK_END_ISO       = "2026-06-08T23:59:00+00:00"
TRADING_WEEK_END_EXCLUSIVE_ISO = "2026-06-09T00:00:00+00:00"
DOWNLOAD_START_ISO         = CALIBRATION_WEEK_START_ISO
DOWNLOAD_END_EXCLUSIVE_ISO = TRADING_WEEK_END_EXCLUSIVE_ISO

# ── Candle interval ────────────────────────────────────────────────────────────
DEFAULT_KLINE_INTERVAL = "1m"
KLINE_INTERVAL_MINUTES = 1

# ── Execution parameters ───────────────────────────────────────────────────────
CAPITAL_SPLIT_SPOT    = 0.5
CAPITAL_SPLIT_FUTURES = 0.5
MIN_WEEKLY_COVERAGE_RATIO = 0.60  # relaxed from 0.98 — illiquid spot pairs have gaps
PAIR_CANDIDATE_COUNT: int | None = None  # None = use all overlapping pairs
MAX_PARALLEL_DOWNLOADS = 3

# ── Fee model ──────────────────────────────────────────────────────────────────
SPOT_TAKER_FEE    = 0.0026   # 0.26%
SPOT_MAKER_FEE    = 0.0016   # 0.16%
FUTURES_TAKER_FEE = 0.0005   # 0.05%
FUTURES_MAKER_FEE = 0.0002   # 0.02%
TAKER_ROUNDTRIP_BPS = 2.0 * (SPOT_TAKER_FEE + FUTURES_TAKER_FEE) * 10_000  # 62 bps
MAKER_ROUNDTRIP_BPS = 2.0 * (SPOT_MAKER_FEE + FUTURES_MAKER_FEE) * 10_000  # 36 bps

print(f"Download window : {DOWNLOAD_START_ISO} → {DOWNLOAD_END_EXCLUSIVE_ISO}")
print(f"Candle interval : {DEFAULT_KLINE_INTERVAL}")
print(f"Taker round-trip: {TAKER_ROUNDTRIP_BPS:.1f} bps  |  Maker round-trip: {MAKER_ROUNDTRIP_BPS:.1f} bps")
print(f"Coverage filter : {MIN_WEEKLY_COVERAGE_RATIO:.0%}")
print(f"Cache directory : {DATA_DIR}")

In [13]:
# ── Time/interval helpers ──────────────────────────────────────────────────────

def iso_to_ts(value: str) -> pd.Timestamp:
    return pd.Timestamp(value)

def iso_to_unix_ms(value: str) -> int:
    return int(pd.Timestamp(value).timestamp() * 1000)

def iso_to_unix_s(value: str) -> int:
    return int(pd.Timestamp(value).timestamp())

def interval_ms(interval_str: str) -> int:
    unit  = interval_str[-1]
    value = int(interval_str[:-1])
    return value * {"m": 60_000, "h": 3_600_000, "d": 86_400_000}[unit]

def expected_rows(start_iso: str, end_exclusive_iso: str, interval_str: str) -> int:
    delta = iso_to_ts(end_exclusive_iso) - iso_to_ts(start_iso)
    return int(delta // pd.Timedelta(milliseconds=interval_ms(interval_str)))


# ── HTTP session ───────────────────────────────────────────────────────────────
session = requests.Session()
session.headers.update({"User-Agent": "alphakit-kraken-arb/0.1"})

MAX_RETRIES = 5
RETRY_SLEEP = 2.0
SPOT_REQUEST_DELAY = 0.6   # Kraken spot: ~1 req/s
FUTURES_REQUEST_DELAY = 0.2


def kraken_get(base_url: str, path: str, params: dict | None = None) -> Any:
    url = f"{base_url}{path}"
    last_exc: Exception | None = None
    for attempt in range(MAX_RETRIES):
        try:
            resp = session.get(url, params=params, timeout=30)
            if resp.status_code == 429:
                time.sleep(RETRY_SLEEP * (attempt + 1))
                continue
            resp.raise_for_status()
            return resp.json()
        except requests.HTTPError as exc:
            last_exc = exc
            if exc.response is not None and exc.response.status_code == 429:
                time.sleep(RETRY_SLEEP * (attempt + 1))
                continue
            raise
        except requests.RequestException as exc:
            last_exc = exc
            time.sleep(RETRY_SLEEP * (attempt + 1))
    if last_exc:
        raise last_exc
    raise RuntimeError("Request failed without exception")


print("Helpers loaded.")

Helpers loaded.


In [14]:
# ── 1. Discover tradeable pairs ─────────────────────────────────────────────────

def _normalise_asset(asset: str) -> str:
    """Strip the X/Z prefix Kraken adds to some asset codes."""
    if len(asset) == 4 and asset[0] in ("X", "Z"):
        return asset[1:]
    return asset


def get_spot_pairs() -> pd.DataFrame:
    payload = kraken_get(KRAKEN_SPOT_BASE, "/AssetPairs")
    if payload.get("error"):
        raise ValueError(f"Kraken spot error: {payload['error']}")
    rows: list[dict] = []
    for internal, info in payload["result"].items():
        if info.get("status") != "online":
            continue
        wsname  = info.get("wsname", "")
        altname = info.get("altname", "")
        base    = info.get("base", "")
        quote   = info.get("quote", "")
        rows.append({
            "internal": internal,
            "altname":  altname,
            "wsname":   wsname,
            "base":     base,
            "quote":    quote,
            "base_norm":  _normalise_asset(base),
            "quote_norm": _normalise_asset(quote),
        })
    return pd.DataFrame(rows)


def get_spot_tickers(altnames: list[str]) -> pd.DataFrame:
    """Fetch 24-h tickers for a list of pair altnames."""
    # Kraken Ticker accepts comma-separated pairs
    chunk_size = 50
    rows: list[dict] = []
    for i in range(0, len(altnames), chunk_size):
        chunk = altnames[i : i + chunk_size]
        payload = kraken_get(KRAKEN_SPOT_BASE, "/Ticker", {"pair": ",".join(chunk)})
        if payload.get("error"):
            continue
        for internal_key, data in payload["result"].items():
            # v[1] = 24h volume in base currency, c[0] = last price
            rows.append({
                "ticker_key": internal_key,
                "spot_last_price": float(data["c"][0]),
                "spot_base_volume_24h": float(data["v"][1]),
            })
        time.sleep(SPOT_REQUEST_DELAY)
    return pd.DataFrame(rows)


def get_futures_instruments() -> pd.DataFrame:
    payload = kraken_get(KRAKEN_FUTURES_BASE, "/instruments")
    rows: list[dict] = []
    for inst in payload.get("instruments", []):
        symbol = inst.get("symbol", "")
        if not inst.get("tradeable", False):
            continue
        # PF_ = perpetual flex (USD-margined), PI_ = perpetual inverse
        if not symbol.startswith(("PF_", "PI_")):
            continue
        rows.append({
            "futures_symbol": symbol,
            "futures_type": "usd_perpetual" if symbol.startswith("PF_") else "inverse_perpetual",
            "underlying": symbol[3:],   # strip PF_ / PI_
            "tag": inst.get("tag", ""),
        })
    return pd.DataFrame(rows)


def get_futures_tickers() -> pd.DataFrame:
    payload = kraken_get(KRAKEN_FUTURES_BASE, "/tickers")
    rows: list[dict] = []
    for t in payload.get("tickers", []):
        rows.append({
            "futures_symbol": t.get("symbol", ""),
            "futures_last_price": float(t.get("markPrice") or t.get("last") or 0),
            "futures_vol24h": float(t.get("vol24h") or 0),
        })
    return pd.DataFrame(rows)


print("Fetching Kraken spot pairs ...")
spot_pairs = get_spot_pairs()
print(f"  Spot pairs online: {len(spot_pairs):,}")

print("Fetching Kraken Futures perpetuals ...")
futures_instruments = get_futures_instruments()
print(f"  Futures perpetuals: {len(futures_instruments):,}")

print("Fetching Futures tickers ...")
futures_tickers = get_futures_tickers()
print(f"  Futures tickers: {len(futures_tickers):,}")

# Focus on USD-margined perpetuals (PF_) that have a spot counterpart
pf_futures = futures_instruments[futures_instruments["futures_type"] == "usd_perpetual"].copy()
pf_futures = pf_futures.merge(futures_tickers, on="futures_symbol", how="left")

# Match: futures underlying (e.g. 'XBTUSD') == spot altname (e.g. 'XBTUSD')
overlap = spot_pairs.merge(
    pf_futures,
    left_on="altname",
    right_on="underlying",
    how="inner",
)

# Filter to USD quote only
overlap = overlap[overlap["quote_norm"] == "USD"].copy()
overlap = overlap.sort_values("futures_vol24h", ascending=False).reset_index(drop=True)

effective_pair_count = len(overlap) if PAIR_CANDIDATE_COUNT is None else min(PAIR_CANDIDATE_COUNT, len(overlap))
pair_candidates = overlap.head(effective_pair_count).copy()

print(f"\nSpot/perpetual overlaps (USD quote): {len(overlap):,}")
print(f"Candidates to evaluate: {effective_pair_count:,}")
display(
    pair_candidates[[
        "altname", "futures_symbol", "base_norm", "quote_norm",
        "futures_last_price", "futures_vol24h",
    ]].head(20)
)

Fetching Kraken spot pairs ...
  Spot pairs online: 1,461
Fetching Kraken Futures perpetuals ...
  Futures perpetuals: 312
Fetching Futures tickers ...
  Futures tickers: 334

Spot/perpetual overlaps (USD quote): 249
Candidates to evaluate: 249


,altname,futures_symbol,base_norm,quote_norm,futures_last_price,futures_vol24h
0,PEPEUSD,PF_PEPEUSD,PEPE,USD,2.771660e-06,3.001449e+11
1,SHIBUSD,PF_SHIBUSD,SHIB,USD,4.666570e-06,1.927005e+10
2,BONKUSD,PF_BONKUSD,BONK,USD,4.354410e-06,3.976409e+09
3,MOGUSD,PF_MOGUSD,MOG,USD,1.074200e-07,1.905015e+09
4,FLOKIUSD,PF_FLOKIUSD,FLOKI,USD,2.424582e-05,7.189150e+08
5,CATUSD,PF_CATUSD,CAT,USD,1.505460e-06,2.964950e+08
6,DOGSUSD,PF_DOGSUSD,DOGS,USD,3.989367e-05,2.309058e+08
7,GALAUSD,PF_GALAUSD,GALA,USD,2.577240e-03,1.700182e+08
8,MEWUSD,PF_MEWUSD,MEW,USD,3.536111e-04,1.156732e+08
9,PUMPUSD,PF_PUMPUSD,PUMP,USD,1.528472e-03,1.072357e+08


In [15]:
# ── 2. OHLC download + parquet cache ───────────────────────────────────────────

def cache_path(symbol: str, market_type: str, interval: str,
               start_iso: str, end_iso: str) -> Path:
    safe_start = start_iso.replace(":", "-")
    safe_end   = end_iso.replace(":", "-")
    return DATA_DIR / f"{symbol}__{market_type}__{interval}__{safe_start}__{safe_end}.parquet"


def validate_frame(frame: pd.DataFrame, symbol: str, market_type: str,
                   interval: str, start_iso: str, end_iso: str) -> bool:
    required = {"timestamp", "symbol", "market_type", "interval",
                "open", "high", "low", "close", "volume"}
    if frame.empty or not required.issubset(frame.columns):
        return False
    ts = pd.to_datetime(frame["timestamp"], utc=True)
    if ts.iloc[0] < iso_to_ts(start_iso):
        return False
    if ts.iloc[-1] >= iso_to_ts(end_iso):
        return False
    if frame["symbol"].iloc[0] != symbol:
        return False
    if frame["market_type"].iloc[0] != market_type:
        return False
    return True


def kline_resample_freq(interval_str: str) -> str:
    """Convert '1m' → '1min', '1h' → '1h' for pandas resample."""
    unit  = interval_str[-1]
    value = interval_str[:-1]
    return f"{value}{'min' if unit == 'm' else 'h'}"


# ── Kraken Spot OHLC (≥ 1 h only — max 720 candles retained per interval) ──────

def fetch_spot_klines(
    altname: str,
    start_iso: str = DOWNLOAD_START_ISO,
    end_exclusive_iso: str = DOWNLOAD_END_EXCLUSIVE_ISO,
) -> pd.DataFrame:
    """
    Fetch spot OHLC from Kraken /OHLC.
    Only use for KLINE_INTERVAL_MINUTES >= 60; finer intervals use fetch_spot_trades_as_ohlc.
    """
    since_ts = iso_to_unix_s(start_iso)
    end_ts   = iso_to_unix_s(end_exclusive_iso)

    all_rows: list[dict] = []
    prev_last = None

    while since_ts < end_ts:
        payload = kraken_get(
            KRAKEN_SPOT_BASE, "/OHLC",
            {"pair": altname, "interval": KLINE_INTERVAL_MINUTES, "since": since_ts},
        )
        if payload.get("error"):
            raise ValueError(f"Kraken spot OHLC error for {altname}: {payload['error']}")

        result   = payload["result"]
        last_ts  = int(result["last"])
        pair_key = next(k for k in result if k != "last")
        candles  = result[pair_key]

        if not candles:
            break

        for c in candles:
            ts_s = int(c[0])
            if ts_s >= end_ts:
                break
            all_rows.append({
                "timestamp": pd.Timestamp(ts_s, unit="s", tz="UTC"),
                "open":   float(c[1]),
                "high":   float(c[2]),
                "low":    float(c[3]),
                "close":  float(c[4]),
                "volume": float(c[6]),
            })

        if last_ts <= since_ts or last_ts == prev_last:
            break
        prev_last = last_ts
        since_ts  = last_ts
        time.sleep(SPOT_REQUEST_DELAY)

    if not all_rows:
        return pd.DataFrame()

    df = pd.DataFrame(all_rows)
    df = df[df["timestamp"] < iso_to_ts(end_exclusive_iso)]
    df["symbol"]      = altname
    df["market_type"] = "spot"
    df["interval"]    = DEFAULT_KLINE_INTERVAL
    return df.drop_duplicates("timestamp").sort_values("timestamp").reset_index(drop=True)


# ── Kraken Spot OHLC from Trades (any interval, full history) ──────────────────

# Kraken's /Trades endpoint returns the full tick history.
# Rate: counter +1 per call, decays 1/s; limit 15. With MAX_PARALLEL_DOWNLOADS=3
# keep TRADES_REQUEST_DELAY >= 1.0 s to stay under ~1 effective req/s per worker.
TRADES_REQUEST_DELAY = 1.2

def fetch_spot_trades_as_ohlc(
    altname: str,
    start_iso: str = DOWNLOAD_START_ISO,
    end_exclusive_iso: str = DOWNLOAD_END_EXCLUSIVE_ISO,
) -> pd.DataFrame:
    """
    Reconstruct spot OHLC by paginating through /Trades and resampling.
    The `since` param is a nanosecond Unix timestamp; `last` in the response
    is the nanosecond cursor for the next page.
    """
    start_s  = iso_to_unix_s(start_iso)
    end_s    = iso_to_unix_s(end_exclusive_iso)
    since_ns = str(start_s * 1_000_000_000)

    prices:  list[float] = []
    volumes: list[float] = []
    times:   list[float] = []

    while True:
        payload = kraken_get(
            KRAKEN_SPOT_BASE, "/Trades",
            {"pair": altname, "since": since_ns, "count": 1000},
        )
        if payload.get("error"):
            raise ValueError(f"Kraken Trades error for {altname}: {payload['error']}")

        result   = payload["result"]
        next_ns  = str(result["last"])
        pair_key = next(k for k in result if k != "last")
        trades   = result[pair_key]

        if not trades:
            break

        done = False
        for t in trades:
            ts = float(t[2])
            if ts >= end_s:
                done = True
                break
            prices.append(float(t[0]))
            volumes.append(float(t[1]))
            times.append(ts)

        if done or next_ns == since_ns:
            break
        since_ns = next_ns
        time.sleep(TRADES_REQUEST_DELAY)

    if not prices:
        return pd.DataFrame()

    freq = kline_resample_freq(DEFAULT_KLINE_INTERVAL)
    idx  = pd.DatetimeIndex(pd.to_datetime(times, unit="s", utc=True), name="timestamp")
    s_price  = pd.Series(prices,  index=idx, dtype=float)
    s_volume = pd.Series(volumes, index=idx, name="volume", dtype=float)

    ohlc = s_price.resample(freq, label="left", closed="left").ohlc()
    vol  = s_volume.resample(freq, label="left", closed="left").sum()
    df   = ohlc.join(vol).dropna(subset=["close"])
    df.index.name = "timestamp"
    df = df.reset_index()

    df = df[
        (df["timestamp"] >= iso_to_ts(start_iso)) &
        (df["timestamp"] <  iso_to_ts(end_exclusive_iso))
    ]
    df["symbol"]      = altname
    df["market_type"] = "spot"
    df["interval"]    = DEFAULT_KLINE_INTERVAL
    return df.drop_duplicates("timestamp").sort_values("timestamp").reset_index(drop=True)


# ── Kraken Futures OHLC ────────────────────────────────────────────────────────

def fetch_futures_klines(
    futures_symbol: str,
    start_iso: str = DOWNLOAD_START_ISO,
    end_exclusive_iso: str = DOWNLOAD_END_EXCLUSIVE_ISO,
) -> pd.DataFrame:
    """
    Fetch futures OHLC from Kraken Futures charts API.

    Endpoint: GET /api/charts/v1/trade/{symbol}/{resolution}
    Params:   from, to  (Unix seconds)
    Response: {"candles": [{"time": ms, ...}], "more_candles": bool}
    """
    from_s     = iso_to_unix_s(start_iso)
    to_s       = iso_to_unix_s(end_exclusive_iso)
    interval_s = KLINE_INTERVAL_MINUTES * 60

    all_rows: list[dict] = []
    cursor = from_s

    while cursor < to_s:
        try:
            resp = session.get(
                f"https://futures.kraken.com/api/charts/v1/trade/{futures_symbol}/{DEFAULT_KLINE_INTERVAL}",
                params={"from": cursor, "to": to_s},
                timeout=30,
            )
            resp.raise_for_status()
        except requests.RequestException as exc:
            raise ValueError(
                f"Kraken Futures OHLCV request failed for {futures_symbol}: {exc}"
            ) from exc

        payload = resp.json()
        candles = payload.get("candles", [])
        if not candles:
            break

        for c in candles:
            ts_ms = int(c["time"])
            ts_s  = ts_ms // 1000
            if ts_s >= to_s:
                break
            all_rows.append({
                "timestamp": pd.Timestamp(ts_ms, unit="ms", tz="UTC"),
                "open":   float(c["open"]),
                "high":   float(c["high"]),
                "low":    float(c["low"]),
                "close":  float(c["close"]),
                "volume": float(c.get("volume", 0)),
            })

        if not payload.get("more_candles", False):
            break
        cursor = int(candles[-1]["time"]) // 1000 + interval_s
        time.sleep(FUTURES_REQUEST_DELAY)

    if not all_rows:
        return pd.DataFrame()

    df = pd.DataFrame(all_rows)
    df = df[df["timestamp"] < iso_to_ts(end_exclusive_iso)]
    df["symbol"]      = futures_symbol
    df["market_type"] = "usd_perpetual"
    df["interval"]    = DEFAULT_KLINE_INTERVAL
    return df.drop_duplicates("timestamp").sort_values("timestamp").reset_index(drop=True)


# ── Load-or-download helper ────────────────────────────────────────────────────

def load_or_download(
    symbol: str,
    market_type: str,
    start_iso: str = DOWNLOAD_START_ISO,
    end_iso: str = DOWNLOAD_END_EXCLUSIVE_ISO,
) -> tuple[pd.DataFrame, str]:
    path = cache_path(symbol, market_type, DEFAULT_KLINE_INTERVAL, start_iso, end_iso)
    if path.exists():
        cached = pd.read_parquet(path)
        if validate_frame(cached, symbol, market_type, DEFAULT_KLINE_INTERVAL, start_iso, end_iso):
            cached["timestamp"] = pd.to_datetime(cached["timestamp"], utc=True)
            return cached.sort_values("timestamp").reset_index(drop=True), "cache"

    if market_type == "spot":
        # OHLC endpoint only has enough history for >= 1 h intervals
        if KLINE_INTERVAL_MINUTES >= 60:
            df = fetch_spot_klines(symbol, start_iso, end_iso)
        else:
            df = fetch_spot_trades_as_ohlc(symbol, start_iso, end_iso)
    elif market_type == "usd_perpetual":
        df = fetch_futures_klines(symbol, start_iso, end_iso)
    else:
        raise ValueError(f"Unknown market_type: {market_type}")

    if df.empty:
        raise ValueError(f"No data downloaded for {symbol} {market_type}")
    df.to_parquet(path, index=False)
    return df, "downloaded"


print("Download functions defined.")

Download functions defined.


In [ ]:
# ── 3. Download candles for all pair candidates ────────────────────────────────

expected_download_rows = expected_rows(
    DOWNLOAD_START_ISO, DOWNLOAD_END_EXCLUSIVE_ISO, DEFAULT_KLINE_INTERVAL
)
print(f"Expected rows per market : {expected_download_rows:,}")
print(f"Pair candidates          : {len(pair_candidates):,}")
print(f"Markets to fetch         : {len(pair_candidates) * 2:,} (spot + futures each)")
print(f"Parallel downloads       : {MAX_PARALLEL_DOWNLOADS}")


def _download_market(symbol: str, market_type: str) -> tuple[tuple[str, str], pd.DataFrame, str]:
    df, src = load_or_download(symbol, market_type)
    return (symbol, market_type), df, src


market_candles: dict[tuple[str, str], pd.DataFrame] = {}
cache_stats = {"cache": 0, "downloaded": 0}
errors: list[str] = []

download_tasks: list[tuple[str, str]] = []
for _row in pair_candidates.itertuples(index=False):
    download_tasks.append((_row.altname, "spot"))
    download_tasks.append((_row.futures_symbol, "usd_perpetual"))

with ThreadPoolExecutor(max_workers=MAX_PARALLEL_DOWNLOADS) as executor:
    futures_map = {
        executor.submit(_download_market, sym, mtype): (sym, mtype)
        for sym, mtype in download_tasks
    }
    done = 0
    total = len(futures_map)
    for fut in as_completed(futures_map):
        done += 1
        sym, mtype = futures_map[fut]
        try:
            key, df, src = fut.result()
            market_candles[key] = df
            cache_stats[src] += 1
        except Exception as exc:
            errors.append(f"{sym} ({mtype}): {exc}")
        if done % 20 == 0 or done == total:
            print(f"  Completed {done}/{total} markets")

if errors:
    print(f"\n⚠️  {len(errors)} errors:")
    for e in errors[:20]:
        print(f"  {e}")

print(f"\nLoaded from cache : {cache_stats['cache']:,}")
print(f"Downloaded now    : {cache_stats['downloaded']:,}")

_coverage_rows = [
    {
        "symbol":      sym,
        "market_type": mtype,
        "rows":        len(df),
        "start":       df["timestamp"].min() if not df.empty else pd.NaT,
        "end":         df["timestamp"].max() if not df.empty else pd.NaT,
    }
    for (sym, mtype), df in market_candles.items()
]
candle_coverage = pd.DataFrame(_coverage_rows)
if not candle_coverage.empty:
    candle_coverage = candle_coverage.sort_values(["market_type", "symbol"]).reset_index(drop=True)

display(candle_coverage.head(20))

Expected rows per market : 20,160
Pair candidates          : 249
Markets to fetch         : 498 (spot + futures each)
Parallel downloads       : 3


In [ ]:
# ── 4. Build spread series and calibrate thresholds ───────────────────────────

def build_pair_frame(
    spot_altname: str,
    futures_symbol: str,
    base_asset: str,
    market_candles: dict,
) -> pd.DataFrame:
    spot = market_candles.get((spot_altname, "spot"), pd.DataFrame()).copy()
    perp = market_candles.get((futures_symbol, "usd_perpetual"), pd.DataFrame()).copy()
    if spot.empty or perp.empty:
        return pd.DataFrame()

    merged = (
        spot[["timestamp", "close"]].rename(columns={"close": "spot_close"})
        .merge(
            perp[["timestamp", "close"]].rename(columns={"close": "perp_close"}),
            on="timestamp", how="inner",
        )
    )
    if merged.empty:
        return pd.DataFrame()

    merged["spot_altname"]    = spot_altname
    merged["futures_symbol"]  = futures_symbol
    merged["base_asset"]      = base_asset
    merged["spread_bps"] = (
        (merged["perp_close"] - merged["spot_close"]) / merged["spot_close"] * 10_000
    )
    merged["week_bucket"] = np.where(
        merged["timestamp"] < iso_to_ts(CALIBRATION_WEEK_END_EXCLUSIVE_ISO),
        "calibration",
        "trading",
    )

    cal_expected   = expected_rows(CALIBRATION_WEEK_START_ISO, CALIBRATION_WEEK_END_EXCLUSIVE_ISO, DEFAULT_KLINE_INTERVAL)
    trade_expected = expected_rows(TRADING_WEEK_START_ISO,     TRADING_WEEK_END_EXCLUSIVE_ISO,     DEFAULT_KLINE_INTERVAL)
    cal_rows   = (merged["week_bucket"] == "calibration").sum()
    trade_rows = (merged["week_bucket"] == "trading").sum()

    if cal_rows < cal_expected * MIN_WEEKLY_COVERAGE_RATIO:
        return pd.DataFrame()
    if trade_rows < trade_expected * MIN_WEEKLY_COVERAGE_RATIO:
        return pd.DataFrame()

    return merged.sort_values("timestamp").reset_index(drop=True)


def simulate_threshold_strategy(
    frame: pd.DataFrame,
    entry_threshold_bps: float,
    target_spread_bps: float,
    starting_capital: float = 1_000.0,
) -> tuple[dict, pd.DataFrame]:
    balance   = starting_capital
    entry_idx = None
    trades: list[dict] = []
    working = frame.sort_values("timestamp").reset_index(drop=True)

    for idx, row in working.iterrows():
        spread = float(row["spread_bps"])
        if entry_idx is None:
            if spread >= entry_threshold_bps:
                entry_idx = idx
            continue

        is_last     = idx == len(working) - 1
        should_exit = spread <= target_spread_bps or is_last
        if not should_exit:
            continue

        entry     = working.iloc[entry_idx]
        spot_cap  = balance * CAPITAL_SPLIT_SPOT
        perp_cap  = balance * CAPITAL_SPLIT_FUTURES
        spot_qty  = spot_cap  / float(entry["spot_close"]) if float(entry["spot_close"]) else 0
        perp_qty  = perp_cap  / float(entry["perp_close"]) if float(entry["perp_close"]) else 0

        spot_pnl = spot_qty * (float(row["spot_close"]) - float(entry["spot_close"]))
        perp_pnl = perp_qty * (float(entry["perp_close"]) - float(row["perp_close"]))
        gross    = spot_pnl + perp_pnl

        # Maker fees for limit-order entry and exit
        fees = (
            spot_cap  * SPOT_MAKER_FEE
            + spot_qty * float(row["spot_close"]) * SPOT_MAKER_FEE
            + perp_cap  * FUTURES_MAKER_FEE
            + perp_qty * float(row["perp_close"]) * FUTURES_MAKER_FEE
        )
        net     = gross - fees
        balance += net

        holding = row["timestamp"] - entry["timestamp"]
        trades.append({
            "entry_time":           entry["timestamp"],
            "exit_time":            row["timestamp"],
            "entry_spread_bps":     float(entry["spread_bps"]),
            "exit_spread_bps":      spread,
            "entry_threshold_bps":  entry_threshold_bps,
            "target_spread_bps":    target_spread_bps,
            "gross_pnl_usd":        gross,
            "fees_usd":             fees,
            "net_pnl_usd":          net,
            "ending_balance":       balance,
            "holding_minutes":      holding.total_seconds() / 60,
            "winner":               net > 0,
        })
        entry_idx = None

    trades_df = pd.DataFrame(trades)
    n = len(trades_df)
    summary = {
        "trade_count":         n,
        "ending_balance_usd":  balance,
        "net_pnl_usd":         balance - starting_capital,
        "return_pct":          (balance / starting_capital - 1.0) * 100,
        "winner_count":        int(trades_df["winner"].sum()) if n else 0,
        "loser_count":         n - int(trades_df["winner"].sum()) if n else 0,
        "win_rate_pct":        float(trades_df["winner"].mean() * 100) if n else 0.0,
        "avg_trade_pnl_usd":   float(trades_df["net_pnl_usd"].mean()) if n else 0.0,
        "avg_holding_minutes": float(trades_df["holding_minutes"].mean()) if n else 0.0,
    }
    return summary, trades_df


def build_threshold_grid(
    calibration: pd.DataFrame,
    target_spread_bps: float,
    fee_bps: float,
    grid_size: int = 25,
) -> np.ndarray:
    profitable = calibration.loc[
        calibration["spread_bps"] >= target_spread_bps + fee_bps,
        "spread_bps",
    ]
    if profitable.empty:
        return np.array([], dtype=float)
    grid = np.quantile(profitable.to_numpy(), np.linspace(0.0, 1.0, grid_size))
    grid = np.unique(np.round(grid, 6))
    return grid[grid >= target_spread_bps + fee_bps]


print("Spread + calibration functions defined.")

In [ ]:
# ── 5. Run threshold optimisation across all pairs ─────────────────────────────

summary_rows: list[dict] = []
pair_series:  dict[str, pd.DataFrame] = {}
threshold_logs: dict[str, pd.DataFrame] = {}

for row in pair_candidates.itertuples(index=False):
    merged = build_pair_frame(
        spot_altname=row.altname,
        futures_symbol=row.futures_symbol,
        base_asset=row.base_norm,
        market_candles=market_candles,
    )
    if merged.empty:
        continue

    calibration = merged[merged["week_bucket"] == "calibration"].copy()
    trading     = merged[merged["week_bucket"] == "trading"].copy()
    if calibration.empty or trading.empty:
        continue

    target_spread = float(calibration["spread_bps"].mean())
    cal_std       = float(calibration["spread_bps"].std(ddof=0))
    threshold_grid = build_threshold_grid(calibration, target_spread, MAKER_ROUNDTRIP_BPS)

    merged["week_target_spread_bps"] = target_spread
    pair_series[row.altname] = merged

    if threshold_grid.size == 0:
        summary_rows.append({
            "spot_altname": row.altname,
            "futures_symbol": row.futures_symbol,
            "base_asset": row.base_norm,
            "target_spread_bps": target_spread,
            "optimal_entry_threshold_bps": np.nan,
            "calibration_trade_count": 0,
            "calibration_return_pct": 0.0,
            "calibration_net_pnl_usd": 0.0,
            "calibration_max_spread_bps": float(calibration["spread_bps"].max()),
            "calibration_mean_spread_bps": target_spread,
            "calibration_std_spread_bps": cal_std,
            "trading_trade_count": 0,
            "trading_return_pct": 0.0,
            "trading_net_pnl_usd": 0.0,
            "trading_win_rate_pct": 0.0,
            "trading_avg_holding_minutes": 0.0,
        })
        threshold_logs[row.altname] = pd.DataFrame()
        continue

    best_threshold = float(threshold_grid[0])
    best_score     = -float("inf")
    best_cal_summary = None
    opt_rows: list[dict] = []

    for t in threshold_grid:
        cal_summary, _ = simulate_threshold_strategy(calibration, float(t), target_spread)
        score = cal_summary["net_pnl_usd"] if cal_summary["trade_count"] > 0 else -float("inf")
        opt_rows.append({"entry_threshold_bps": float(t), **cal_summary})
        if score > best_score:
            best_score = score
            best_threshold = float(t)
            best_cal_summary = cal_summary

    trading_summary, _ = simulate_threshold_strategy(trading, best_threshold, target_spread)
    threshold_logs[row.altname] = pd.DataFrame(opt_rows).sort_values("net_pnl_usd", ascending=False)

    summary_rows.append({
        "spot_altname": row.altname,
        "futures_symbol": row.futures_symbol,
        "base_asset": row.base_norm,
        "target_spread_bps": target_spread,
        "optimal_entry_threshold_bps": best_threshold,
        "calibration_trade_count": best_cal_summary["trade_count"] if best_cal_summary else 0,
        "calibration_return_pct": best_cal_summary["return_pct"] if best_cal_summary else 0.0,
        "calibration_net_pnl_usd": best_cal_summary["net_pnl_usd"] if best_cal_summary else 0.0,
        "calibration_max_spread_bps": float(calibration["spread_bps"].max()),
        "calibration_mean_spread_bps": target_spread,
        "calibration_std_spread_bps": cal_std,
        "trading_trade_count": trading_summary["trade_count"],
        "trading_return_pct": trading_summary["return_pct"],
        "trading_net_pnl_usd": trading_summary["net_pnl_usd"],
        "trading_win_rate_pct": trading_summary["win_rate_pct"],
        "trading_avg_holding_minutes": trading_summary["avg_holding_minutes"],
    })

pair_summary = (
    pd.DataFrame(summary_rows)
    .sort_values(["trading_net_pnl_usd", "calibration_net_pnl_usd"], ascending=[False, False])
    if summary_rows
    else pd.DataFrame()
)

print("Pair threshold optimisation complete.")
display(pair_summary)

In [ ]:
# ── 6. Out-of-sample trading-week backtest ─────────────────────────────────────

STARTING_CAPITAL = 1_000.0

trade_summaries: list[dict] = []
trade_logs: dict[str, pd.DataFrame] = {}

for row in pair_summary.itertuples(index=False):
    if pd.isna(row.optimal_entry_threshold_bps):
        trade_logs[row.spot_altname] = pd.DataFrame()
        trade_summaries.append({
            "spot_altname": row.spot_altname,
            "futures_symbol": row.futures_symbol,
            "optimal_entry_threshold_bps": np.nan,
            "target_spread_bps": row.target_spread_bps,
            "starting_balance_usd": STARTING_CAPITAL,
            "ending_balance_usd": STARTING_CAPITAL,
            "net_pnl_usd": 0.0,
            "return_pct": 0.0,
            "trade_count": 0,
            "winner_count": 0,
            "win_rate_pct": 0.0,
            "avg_holding_minutes": 0.0,
        })
        continue

    trading = pair_series[row.spot_altname]
    trading = trading[trading["week_bucket"] == "trading"].copy()
    summary, trades_df = simulate_threshold_strategy(
        trading,
        entry_threshold_bps=float(row.optimal_entry_threshold_bps),
        target_spread_bps=float(row.target_spread_bps),
        starting_capital=STARTING_CAPITAL,
    )
    trade_logs[row.spot_altname] = trades_df
    trade_summaries.append({
        "spot_altname": row.spot_altname,
        "futures_symbol": row.futures_symbol,
        "optimal_entry_threshold_bps": float(row.optimal_entry_threshold_bps),
        "target_spread_bps": float(row.target_spread_bps),
        "starting_balance_usd": STARTING_CAPITAL,
        "ending_balance_usd": summary["ending_balance_usd"],
        "net_pnl_usd": summary["net_pnl_usd"],
        "return_pct": summary["return_pct"],
        "trade_count": summary["trade_count"],
        "winner_count": summary["winner_count"],
        "win_rate_pct": summary["win_rate_pct"],
        "avg_holding_minutes": summary["avg_holding_minutes"],
    })

trade_summary = (
    pd.DataFrame(trade_summaries)
    .sort_values(["ending_balance_usd", "trade_count"], ascending=[False, False])
    if trade_summaries else pd.DataFrame()
)

print("Trading-week backtest complete (taker/taker, 50/50 spot-futures allocation)")
display(trade_summary)

In [ ]:
# ── 7. Visualise top pairs ─────────────────────────────────────────────────────

executed = trade_summary[trade_summary["trade_count"] > 0].copy()
print(f"Pairs with at least one executed trade: {len(executed):,}")
display(executed)

top_n = executed.head(5)
if not top_n.empty:
    fig = make_subplots(
        rows=len(top_n), cols=1,
        shared_xaxes=True,
        subplot_titles=list(top_n["spot_altname"]),
    )
    for i, row in enumerate(top_n.itertuples(), start=1):
        frame   = pair_series[row.spot_altname]
        trading = frame[frame["week_bucket"] == "trading"]
        fig.add_trace(
            go.Scatter(x=trading["timestamp"], y=trading["spread_bps"],
                       mode="lines", name=row.spot_altname, showlegend=False),
            row=i, col=1,
        )
        fig.add_hline(y=float(frame["week_target_spread_bps"].iloc[0]),
                      line_width=1, line_dash="dot",  line_color="gray",    row=i, col=1)
        if not pd.isna(row.optimal_entry_threshold_bps):
            fig.add_hline(y=float(row.optimal_entry_threshold_bps),
                          line_width=1, line_dash="dash", line_color="crimson", row=i, col=1)
    fig.update_layout(
        height=260 * len(top_n), width=1100,
        template="plotly_white",
        title="Trading-week spread bps with calibrated target (dot) and entry threshold (red dash)",
    )
    fig.show()

# Print threshold search log for top 3
for sym in top_n.head(3)["spot_altname"].tolist():
    print(f"\nThreshold search log: {sym}")
    display(threshold_logs.get(sym, pd.DataFrame()).head(10))